In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [7]:
from langchain_community.document_loaders import PyPDFLoader, MergedDataLoader

In [8]:
docs = MergedDataLoader([
    PyPDFLoader("data/GTB_gold_Nov23.pdf"),
    PyPDFLoader("data/GTB_platinum_Nov23.pdf")
]).load()

In [9]:
len(docs)

53

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

slices = splitter.split_documents(docs)

In [12]:
len(slices)

200

In [13]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [14]:
embedding_model = OpenAIEmbeddings()

embedding_model.model

'text-embedding-ada-002'

In [15]:
from langchain_community.vectorstores import InMemoryVectorStore

In [16]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=slices,
    embedding=embedding_model
)

In [17]:
retriever = vectorstore.as_retriever()

In [18]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

In [19]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """Gere cinco consultas de pesquisa para um banco de dados vetorial a partir
     de uma pergunta do usuario, permitindo uma resposta mais precisa para a busca semantica

     Gere somente o texto das consultas separadas por quebra de linha e nada de texto adicional
     """),
    ("human", "{question}")
])

query_model = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.5,
    api_key=openai_api_key
)

rewrite_chain = rewrite_prompt | query_model | CommaSeparatedListOutputParser()

In [20]:
query = "Como lidar com cartão roubado?"

In [21]:
rewrite_chain.invoke(query)

['1. Quais são os passos imediatos ao perceber um cartão roubado?',
 '2. Como bloquear ou cancelar um cartão de crédito roubado?',
 '3. Quais documentos são necessários para solicitar um novo cartão após roubo?',
 '4. Como prevenir o uso indevido de um cartão roubado?',
 '5. Quais medidas a instituição financeira oferece para proteção após roubo de cartão?']

In [22]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [23]:
multi_query_retriever = MultiQueryRetriever(
    retriever=retriever,
    llm_chain=rewrite_chain
)

In [24]:
multi_query_retriever.invoke(query)

[Document(id='0509d33a-ed22-4bf4-9adc-6dbc961f3b7f', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T10:05:20-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:24:39Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '8050f60f-7cb7-4b1b-99e0-7f18410feb72', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T10:05:20-03:00', 'source': 'data/GTB_gold_Nov23.pdf', 'total_pages': 24, 'page': 17, 'page_label': '18'}, page_content='* É possível que seja solicitado ao Portado

In [25]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [26]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responda utilizando exclusivamente o conteúdo fornecido: \n\nContexto: \n{contexto}"),
    ("human", "{query}")
])

In [27]:
from langchain_core.runnables import RunnablePassthrough

In [28]:
multi_query_rag_chain = (
    {
        "contexto": RunnablePassthrough() | multi_query_retriever | format_docs,
        "query": RunnablePassthrough()
    } | rag_prompt | query_model | StrOutputParser()
)

In [29]:
multi_query_rag_chain.invoke(query)

'O conteúdo fornecido não especifica as etapas exatas para lidar com um cartão roubado.'

In [30]:
from langchain_classic.evaluation.qa import QAGenerateChain, QAEvalChain

In [33]:
eval_chain = QAEvalChain.from_llm(query_model)
generate_chain = QAGenerateChain.from_llm(query_model)

In [92]:
def evaluate(qa, generated):
    evaluations = eval_chain.evaluate(qa, generated)

    hit = 0
    for i, item in enumerate(qa):
        hit += (1 if evaluations[i]["results"].split("\n")[-1].split(":")[-1].strip() == "CORRECT" else 0)

    return hit / len(qa)

In [39]:
qa = generate_chain.apply(
    [{ "doc" : p.page_content } for p in slices ]
)

In [40]:
qa

[{'qa_pairs': {'query': 'A partir de que data os benefícios e serviços do Programa de Cartão Mastercard Gold passam a estar disponíveis para os portadores do cartão elegíveis, de acordo com o Guia de Benefícios de novembro de 2023?  ',
   'answer': 'Os benefícios e serviços do Programa de Cartão Mastercard Gold passam a estar disponíveis para os portadores do cartão elegíveis a partir de 1 de novembro de 2023.'}},
 {'qa_pairs': {'query': 'Quais são as entidades envolvidas na emissão do seguro mencionado no documento, e qual é o papel da Mastercard nesse contexto?',
   'answer': 'As entidades envolvidas são a Mastercard do Brasil Ltda., que atua como representante do contrato de seguros; a seguradora AIG Seguros Brasil S.A., que é a seguradora responsável pelo seguro; e a corretora Apolix Corretora de Seguros Ltda. Além disso, a Mastercard figura como mera representante do contrato de seguros e paga os benefícios fornecidos pelos produtos, sem incentivar ou recomendar sua comercializaçã

In [59]:
generated_no_rag = []
generated_rag = []

for q_n_a in qa[:10]:
    generated_no_rag.append(
        {"result" : query_model.invoke(q_n_a["qa_pairs"]["query"]).content}
    )

    generated_rag.append(
        {"result" : multi_query_rag_chain.invoke(q_n_a["qa_pairs"]["query"])}
    )

In [60]:
generated_no_rag

[{'result': 'De acordo com o Guia de Benefícios de novembro de 2023, os benefícios e serviços do Programa de Cartão Mastercard Gold passam a estar disponíveis para os portadores do cartão elegíveis a partir de 1º de novembro de 2023.'},
 {'result': 'Para responder à sua pergunta, preciso de mais informações específicas sobre o documento ao qual você se refere, como o tipo de seguro mencionado e o conteúdo exato do documento. No entanto, de forma geral, em um contexto típico de emissão de seguro associado a um cartão de crédito, as entidades envolvidas costumam ser:\n\n1. **Seguradora**: A empresa responsável por oferecer a cobertura do seguro, assumindo os riscos e pagando indenizações conforme os termos do contrato.\n2. **Emissor do Cartão (Banco ou Instituição Financeira)**: A instituição que emite o cartão de crédito ou débito, muitas vezes responsável por oferecer o seguro como benefício adicional aos seus clientes.\n3. **Mastercard**: A bandeira do cartão, que atua como intermediá

In [61]:
generated_rag

[{'result': 'A partir de 1 de novembro de 2023.'},
 {'result': 'As entidades envolvidas na emissão do seguro mencionado no documento são a AIG Seguros Brasil S.A., que é a seguradora autorizada pela SUSEP a funcionar no Brasil, e a Mastercard do Brasil Ltda., que atua como representante do contrato de seguros. A Mastercard figura como mera representante do contrato de seguros, não sendo a seguradora responsável pelo pagamento das indenizações ou pela administração direta do seguro. Seu papel é atuar como intermediária, disponibilizando os benefícios pagos pela bandeira do cartão e fornecendo informações e suporte aos beneficiários.'},
 {'result': 'De acordo com o documento, o registro do plano de seguros na SUSEP não implica, por parte da autarquia, incentivo ou recomendação de sua comercialização.'},
 {'result': 'Para dar entrada em uma ocorrência ou obter informações sobre os serviços do Mastercard Global Service™, você pode ligar para o número gratuito específico do seu país ou, se 

In [ ]:
qa_copy = [
    {
        "query": item["qa_pairs"]["query"],
        "answer": item["qa_pairs"]["answer"]
    } for item in qa[:10]
]

In [93]:
evaluate(qa_copy, generated_no_rag)

0.3

In [94]:
evaluate(qa_copy, generated_rag)

0.6